# LAB | First match ≠ right match
## Relevance Scoring & Rerankers for Trustworthy AI + EU AI Act

**Learning goal:** build one retrieval path over two sources and **prove** every answer is grounded in evidence you can inspect.

**Your sources**
- `podcast.txt`: a podcast transcript about the EU *Ethics Guidelines for Trustworthy AI* (2019)
- `eu_ai_act.pdf`: Regulation (EU) 2024/1689, the AI Act (144 pages)

**What you submit:** a `lab_proof.md` containing your code (or a link to this notebook), **one query**, the **retrieved evidence**, the **final answer**, and **one failure or limitation**. There's a template at the end of this notebook.

---

### How this notebook works

| Label | Meaning |
|---|---|
| **PROVIDED** | Code is given. Run it and read it. |
| **YOUR TURN** | Replace the `# TODO` lines with your own code. |
| **OPTIONAL (Advanced)** | Do this if you have time; it's not required to pass. |

Each step ends with a **Done when** check. Don't move on until it passes.

### Prompt discipline
You may use AI to **review, debug or critique** your work, but not to write your final answer or your `lab_proof.md`. A good review prompt:

> *Role: Retrieval reviewer. Task: Review my plan for grounding and verification. Inputs: my sources, query, retrieval path, and evidence. Constraints: Do not produce the final answer. Output: three grounding risks, two trace checks, one simplification.*

---
## Step 0: Environment [PROVIDED]

1. Create a `.env` file next to this notebook containing `OPENAI_API_KEY=...` and `PINECONE_API_KEY=...`
2. Create a folder called `data/` next to this notebook and put `podcast.txt` and `eu_ai_act.pdf` inside it.
3. Run the two cells below.

**Warning:** Never paste API keys into a notebook cell, and never commit your `.env` file.

In [1]:
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


---
## Step 1: Setup & Data Preparation [PROVIDED, except 1d]

**Goal:** turn two very different sources into chunks that each carry **metadata**, meaning labels that say *where the chunk came from*.

**Why metadata matters:** when the system answers *"Fines are up to €35M"*, a reviewer must be able to trace that sentence back to *"EU AI Act, page 115"*. Without metadata you can't trace an answer, and you can't filter results either (Step 5).

Parts 1a–1c are done for you: run each cell and **read the output**. Part 1d (embeddings) is yours to write.

### 1a. Load both sources

In [3]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

C:\Users\marth\AppData\Local\Temp\ipykernel_36144\320955083.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'data\\eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


### 1b. Chunk with metadata

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents into chunks and give every chunk a clean, consistent metadata schema."""
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,        # "podcast" or "regulation" -> used for filtering in Step 5
            "source_name": source_name,        # human-readable name -> used for citations
            "chunk_id": f"{source_type}-{i}",  # unique id -> lets a reviewer trace an exact chunk
        }
        # PDF pages are 0-indexed in the loader; store the page number a human would see
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1

        if source_type == "regulation":
            page = metadata["page"]

            if page >= 124:
                metadata["section"] = "annex"
            elif page <= 43:
                metadata["section"] = "recital"
            else:
                metadata["section"] = "article"

            chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


In [35]:
print(act_chunks[0].metadata)
print(podcast_chunks[0].metadata)

{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-0', 'page': 1, 'section': 'recital'}
{'source': 'data\\podcast.txt'}


In [36]:
from collections import Counter

print(Counter(chunk.metadata.get("section") for chunk in act_chunks))

Counter({'article': 396, 'recital': 308, 'annex': 60})


In [31]:
for chunk in act_chunks:
    if chunk.metadata.get("section") == "annex":
        print(chunk.metadata["page"])
        print(chunk.page_content[:150])
        print("-" * 40)

124
ANNEX I
List of Union harmonisation legislation
Section A. List of Union harmonisation legislation based on the New Legislative Framework
1. Directive
----------------------------------------
124
5. Directive 2014/34/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of 
the laws of the Member States relat
----------------------------------------
124
installations and repealing Directive 2000/9/EC (OJ L 81, 31.3.2016, p. 1);
9. Regulation (EU) 2016/425 of the European Parliament and of the Council 
----------------------------------------
124
medical devices and repealing Directive 98/79/EC and Commission Decision 2010/227/EU (OJ L 117, 5.5.2017, 
p. 176).
Section B. List of other Union har
----------------------------------------
125
16. Directive 2014/90/EU of the European Parliament and of the Council of 23 July 2014 on marine equipment and 
repealing Council Directive 96/98/EC (
----------------------------------------
125
requirements for

In [32]:
for chunk in act_chunks:
    if 43 <= chunk.metadata["page"] <= 46:
        print("Page:", chunk.metadata["page"])
        print("Section:", chunk.metadata["section"])
        print(chunk.page_content[:200])
        print("-" * 40)

Page: 43
Section: recital
impact on their health, safety or fundamental rights. That explanation should be clear and meaningful and should 
provide a basis on which the affected persons are able to exercise their rights. The r
----------------------------------------
Page: 43
Section: recital
accordance with Article 290 TFEU should be delegated to the Commission to amend the conditions under which an 
AI system is not to be considered to be high-risk, the list of high-risk AI systems, the 
----------------------------------------
Page: 43
Section: recital
general-purpose AI models with systemic risk, the technical documentation for providers of general-purpose AI 
models and the transparency information for providers of general-purpose AI models. It is
----------------------------------------
Page: 43
Section: recital
(174) Given the rapid technological developments and the technical expertise required to effectively apply this Regulation, 
the Commission should evaluate and review thi

In [33]:
for chunk in act_chunks:
    if "CHAPTER I" in chunk.page_content:
        print("Page:", chunk.metadata["page"])
        print(chunk.page_content[:300])
        print("-" * 40)

Page: 44
with Article 42(1) and (2) of Regulation (EU) 2018/1725 and delivered their joint opinion on 18 June 2021,
HAVE ADOPTED THIS REGULATION:
CHAPTER I
GENERAL PROVISIONS
Article 1
Subject matter`
1. The purpose of this Regulation is to improve the functioning of the internal market and promote the uptak
----------------------------------------
Page: 51
Article 4
AI literacy
Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
their staff and other persons dealing with the operation and use of AI systems on their behalf, taking into account their 
technical knowledge, exper
----------------------------------------
Page: 53
including information on the number of the decisions taken by competent judicial authorities or an independent 
administrative authority whose decision is binding upon requests for authorisations in accordance with paragraph 3 and 
their result.
7. The Commission shall publish annual repo

### 1c. Inspect your chunks (always look at your data!)

In [5]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

{'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'page': 21}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of no ...
--------------------------------------------------------------------------------


### 1d. Embeddings model [YOUR TURN]

Set up the embedding model yourself and test it on **one** chunk.

**What to do**
1. Import `OpenAIEmbeddings` from `langchain_openai`.
2. Create a variable called `embeddings` using the model `text-embedding-3-small`. Step 2 uses this exact variable name.
3. Embed the text of `all_chunks[0]` with `embeddings.embed_query(...)` and print the length of the vector.

**Hint:** see **Section 2 of `rag_intro.ipynb`**.
**Note:** you don't embed **all** chunks here. That happens in Step 2, when you upload them to Pinecone, so you only pay for them once.

In [6]:
# YOUR TURN: set up the embedding model and test it on one chunk

# TODO: import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings

# TODO: embeddings = ...
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector = embeddings.embed_query(all_chunks[0].page_content)

print(len(vector))

# TODO: embed all_chunks[0].page_content and print the vector length

1536


### Stop and think (write your answers in `lab_proof.md` later)

1. Our metadata has `source_type`, `source_name`, `chunk_id` and `page`. **What's missing?** Think about a user asking *"What does Article 5 say?"* or *"Only show me recitals."* Could you filter on that today?
2. The podcast has no pages. How would a reviewer find where a podcast chunk came from?

You'll come back to this in **Step 5**.

**Done when:** you can see chunk counts for both sources, every chunk you print shows a `metadata` dict, and your test embedding has length **1536**.

---
## Step 2: Embeddings + Baseline Retrieval [YOUR TURN]

**Goal:** build a plain vector search. This is your **baseline**: every later step must beat it, or it isn't worth the extra cost.

**What to do**
1. Create (or connect to) a Pinecone index. Use dimension `1536` and metric `cosine`.
2. Upload `all_chunks` with `PineconeVectorStore.from_documents(...)`. This is the step that embeds every chunk.
3. Write `baseline_search(query, k)`. It should return `(doc, score)` pairs using `similarity_search_with_score`.
4. Run it on the **fixed test queries** below and save the results.

**Hint:** Stuck? Look at **Section 3–4 of `rag_intro.ipynb`** (yesterday's notebook).
**Warning:** Only run the upload cell **once**. Running it again uploads duplicate chunks.

In [10]:
# PROVIDED: fixed test queries. Use the SAME queries in every step so you can compare fairly.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

# PROVIDED: helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [38]:
# YOUR TURN: create / connect to the Pinecone index
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers-v2"   # use lowercase letters and dashes only

# TODO: if index_name does not exist yet, create it (dimension=1536, metric="cosine",
#       spec=ServerlessSpec(cloud="aws", region="us-east-1"))
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# TODO: vectorstore = PineconeVectorStore.from_documents(...)
vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [39]:
print(pc.list_indexes().names())

['lab-rerankers', 'lab-rerankers-v2']


In [11]:
# YOUR TURN: baseline retrieval
def baseline_search(query, k=5):
    return vectorstore.similarity_search_with_score(query, k=k)

baseline_results = {}
for q in TEST_QUERIES:
    # TODO: call baseline_search, store the result in baseline_results[q], and print it with show_results
    baseline_results[q] = baseline_search(q)
    show_results(q, baseline_results[q], label="Baseline")



=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.681  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#4  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#5  score=0.660  [regulation | p.3.0]  'operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in \ncompl'

=== Baseline | Query: What are the seven requirements for trustworthy AI?
#1  score=0.674  [podcast | p

**Done when:** each test query prints 5 results with a score, a source type and a page or chunk id.

**Note it down:** for which query does the **top result** look wrong or only half-relevant? That's your *first match ≠ right match* case.

---
## Step 3: LLM Relevance Scoring [OPTIONAL (Advanced)]

**Goal:** have an LLM judge how relevant each retrieved chunk is, then re-order the results.

**What to do**
1. Retrieve a wider set of candidates (e.g. `k=10`) with `baseline_search`.
2. For each chunk, ask `gpt-4o-mini` to score its relevance to the query from 0 to 1.
3. Combine the two scores, e.g. `combined = 0.3 * similarity + 0.7 * llm_score`, and sort by `combined`.
4. Compare the new top 3 with the baseline top 3.

**Hint:** Reuse `rerank_documents()` from **Section 3 of `rag_pipeline.ipynb`**. It already parses the score and prints it.
**Warning:** This makes 1 LLM call **per chunk, per query**. Count the calls you make.

In [13]:
# YOUR TURN
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def llm_relevance_score(query, doc):
    # TODO: prompt the LLM for a 0-1 score, parse it robustly, return a float
    prompt = f"Rate how relevant this document is to the question from 0 to 1. Return only a number.\nQuestion: {query}\nDocument: {doc.page_content}"

    response = llm.invoke(prompt)
    try:
     score = float(response.content.strip())
     return max(0.0, min(1.0, score))
    except (ValueError, TypeError, AttributeError):
     return 0.0

def llm_rerank(query, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7):
    # TODO: retrieve k_initial -> score each chunk -> combine scores -> sort -> return top k_final as (doc, combined_score)
        candidates = baseline_search(query, k=k_initial)
        scored_results = []
        for doc, similarity_score in candidates:
            relevance_score = llm_relevance_score(query, doc)
            combined_score = w_sim * similarity_score + w_llm * relevance_score
            scored_results.append((doc, combined_score))
        scored_results.sort(key=lambda item: item[1], reverse=True)
        return scored_results[:k_final]


In [14]:
query = TEST_QUERIES[3]

print("BEFORE RERANKING")
show_results(query, baseline_results[query], label="Baseline")

print("\nAFTER RERANKING")
reranked_results = llm_rerank(query)
show_results(query, reranked_results, label="LLM Reranked")

BEFORE RERANKING

=== Baseline | Query: What fines apply for using a prohibited AI practice?
#1  score=0.633  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.617  [regulation | p.118.0]  '(d) failed to make available to the Commission access to the general-purpose AI model or general-purpose AI model with \n'
#3  score=0.613  [regulation | p.115.0]  'including start-ups, and their economic viability.\n2. The Member States shall, without delay and at the latest by the da'
#4  score=0.610  [regulation | p.117.0]  'with regard to the same subject matter;\n(e) any similar previous infringements by the Union institution, body, office or'
#5  score=0.584  [regulation | p.117.0]  'undertakings in the protection of their personal data or business secrets.\n6. Funds collected by imposition of fines in '

AFTER RERANKING

=== LLM Reranked | Query: What fines apply for using a prohibite

In [15]:
top_doc = reranked_results[0][0]

print("SOURCE:", top_doc.metadata)
print("\nFULL TEXT:")
print(top_doc.page_content)

SOURCE: {'chunk_id': 'regulation-665', 'page': 115.0, 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}

FULL TEXT:
including start-ups, and their economic viability.
2. The Member States shall, without delay and at the latest by the date of entry into application, notify the Commission 
of the rules on penalties and of other enforcement measures referred to in paragraph 1, and shall notify it, without delay, of 
any subsequent amendment to them.
3. Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative 
fines of up to EUR 35 000 000 or, if the offender is an undertaking, up to 7 % of its total worldwide annual turnover for the 
preceding financial year, whichever is higher.
4. Non-compliance with any of the following provisions related to operators or notified bodies, other than those laid 
down in Articles 5, shall be subject to administrative fines of up to EUR 15 000 000 or, if the offender is an u

**Done when:** you can show at least one query where the order changed, and say whether it got better or worse.

---
## Step 4: Dedicated Reranker (Cross-Encoder or Cohere) [OPTIONAL (Advanced)]

**Goal:** use a model built specifically for reranking. It's usually faster and cheaper than an LLM, and often better.

**Option A: local cross-encoder (no API key needed)**
```
%pip install -q sentence-transformers
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(query, doc.page_content) for doc in docs])
```

**Option B: Cohere Rerank API** (needs `COHERE_API_KEY` in `.env`; see the Cohere Rerank docs)

**What to do:** rerank the same 10 baseline candidates, keep the top 5, and compare with Step 3.

In [16]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

c:\Users\marth\.conda\envs\bootcamp-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marth\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\marth\.conda\envs\bootcamp-env\Lib\site-packages\huggingface_hub\file_download.py:1

In [18]:
# YOUR TURN
def crossencoder_rerank(query, k_initial=10, k_final=5):
    # TODO: retrieve k_initial candidates -> score (query, chunk) pairs -> sort -> return top k_final as (doc, score)
        candidates = baseline_search(query, k=k_initial)

        docs = [doc for doc, score in candidates]

        pairs = [(query, doc.page_content) for doc in docs]

        scores = reranker.predict(pairs)
        ranked_results = sorted(zip(docs, scores), key=lambda item: item[1], reverse=True)
        return ranked_results[:k_final]


In [19]:
query = TEST_QUERIES[3]

print("BEFORE RERANKING")
show_results(query, baseline_results[query], label="Baseline")

print("\nAFTER CROSS-ENCODER RERANKING")
crossencoder_results = crossencoder_rerank(query)
show_results(query, crossencoder_results, label="Cross-Encoder")

BEFORE RERANKING

=== Baseline | Query: What fines apply for using a prohibited AI practice?
#1  score=0.633  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.617  [regulation | p.118.0]  '(d) failed to make available to the Commission access to the general-purpose AI model or general-purpose AI model with \n'
#3  score=0.613  [regulation | p.115.0]  'including start-ups, and their economic viability.\n2. The Member States shall, without delay and at the latest by the da'
#4  score=0.610  [regulation | p.117.0]  'with regard to the same subject matter;\n(e) any similar previous infringements by the Union institution, body, office or'
#5  score=0.584  [regulation | p.117.0]  'undertakings in the protection of their personal data or business secrets.\n6. Funds collected by imposition of fines in '

AFTER CROSS-ENCODER RERANKING

=== Cross-Encoder | Query: What fines apply for us

In [20]:
query2 = TEST_QUERIES[4]

print("BEFORE RERANKING")
show_results(query2, baseline_results[query2], label="Baseline")

print("\nAFTER CROSS-ENCODER RERANKING")
crossencoder_results2 = crossencoder_rerank(query2)
show_results(query2, crossencoder_results2, label="Cross-Encoder")

BEFORE RERANKING

=== Baseline | Query: Is social scoring of citizens allowed?
#1  score=0.588  [regulation | p.9.0]  'multiple data points related to their social behaviour in multiple contexts or known, inferred or predicted personal \nor'
#2  score=0.497  [regulation | p.9.0]  'face or fingerprint, to deduce or infer an individuals’ political opinions, trade union membership, religious or \nphilos'
#3  score=0.497  [regulation | p.51.0]  '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation or classification of'
#4  score=0.485  [podcast | podcast-16]  'learning for every child. The potential is incredible, but the expert group lists some specific critical concerns. These'
#5  score=0.421  [regulation | p.16.0]  'should be granted, denied, reduced, revoked or reclaimed by authorities, including whether beneficiaries are \nlegitimate'

AFTER CROSS-ENCODER RERANKING

=== Cross-Encoder | Query: Is social scoring of citizens allowed?
#1  sc

**Done when:** you have a side-by-side of baseline vs reranked top 3 for at least two queries.

---
## Step 5: Metadata Filtering [YOUR TURN]

**Goal:** restrict search to the right part of the knowledge base *before* ranking.

**What to do**
1. Filter by source using Pinecone's filter syntax:
   `vectorstore.similarity_search_with_score(query, k=5, filter={"source_type": {"$eq": "regulation"}})`
2. Run one query twice: once with **only the regulation**, once with **only the podcast**. How do the answers differ?
3. **The real lesson:** try to filter on something your metadata *doesn't* have yet, such as `section` (`"recital"` / `"article"` / `"annex"`) or `article` (`"Article 5"`). You can't.
   → Go back to `chunk_with_metadata` in Step 1, add the field, **delete the index and re-upload** (`pc.delete_index(index_name)`), then try again.

**Hint:** Hints for the EU AI Act: in this PDF the recitals run up to about page 44, the Articles start at *"CHAPTER I"*, and the Annexes start at *"ANNEX I"*. A regex like `r"^Article\s+(\d+)"` on each line can detect article headings.
**Warning:** Pinecone metadata values can't be `None`. Leave the key out instead.

In [21]:
# YOUR TURN: filter by source_type
# TODO
query = "What AI practices are prohibited under the EU AI Act?"

regulation_results = vectorstore.similarity_search_with_score(
    query,
    k=5,
    filter={"source_type": {"$eq": "regulation"}}
)

podcast_results = vectorstore.similarity_search_with_score(
    query,
    k=5,
    filter={"source_type": {"$eq": "podcast"}}
)

show_results(query, regulation_results, label="Regulation Only")
show_results(query, podcast_results, label="Podcast Only")


=== Regulation Only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.681  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#4  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#5  score=0.660  [regulation | p.3.0]  'operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in \ncompl'

=== Podcast Only | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.519 

In [40]:
results = vectorstore.similarity_search(
    "What AI practices are prohibited?",
    k=3,
    filter={
        "source_type": {"$eq": "regulation"},
        "section": {"$eq": "article"}
    }
)

for doc in results:
    print(doc.metadata)

{'chunk_id': 'regulation-342', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
{'chunk_id': 'regulation-343', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
{'chunk_id': 'regulation-656', 'page': 113.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}


In [41]:
# YOUR TURN: filter by the NEW metadata field you added (section / article / ...)
results = vectorstore.similarity_search(
    "What AI practices are prohibited?",
    k=3,
    filter={
        "source_type": {"$eq": "regulation"},
        "section": {"$eq": "article"}
    }
)

for doc in results:
    print(doc.metadata)

{'chunk_id': 'regulation-342', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
{'chunk_id': 'regulation-343', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
{'chunk_id': 'regulation-656', 'page': 113.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}


**Done when:** you can run a query that only returns, e.g., *Article* chunks from the EU AI Act, and you can explain why you had to rebuild the index to do it.


### Step 5 – Metadata Filtering: Explanation

We rebuilt the Pinecone index because the original index did not contain the `section` metadata needed for filtering.

After modifying the `chunk_with_metadata()` function in Step 1b to classify chunks as `recital`, `article`, or `annex`, we created a new Pinecone index (`lab-rerankers-v2`) and uploaded the updated chunks.

This allowed us to filter search results by both `source_type` and `section`.

**Result:** The search successfully returned three chunks from the EU AI Act, all classified as `article`.

---
## Step 6: Complete RAG Pipeline with Reranking [YOUR TURN]

**Goal:** plug your best retrieval method into a full RAG chain that **cites its sources**.

**What to do**
1. Build an LCEL chain: `retrieve → format context → prompt → LLM → answer` (see **Section 5 of `rag_intro.ipynb`**).
2. In `format_context`, put each chunk's source and page **inside** the context, e.g. `[EU AI Act, p.51]`, and tell the LLM to cite them.
3. Make it easy to switch retrieval methods: baseline vs. reranked should be a **1–2 line change**.
4. Make the model say *"I don't know based on the sources"* when the context doesn't contain the answer.

In [48]:
# YOUR TURN
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def format_context(results):
    # TODO: join chunks into one string, each prefixed with its source + page/chunk_id
    formatted_chunks = []

    for doc, score in results:
        source = doc.metadata.get("source_name", "Unknown source")
        page = doc.metadata.get("page", "N/A")

        text = f"[{source}, p.{page}]\n{doc.page_content}"
        formatted_chunks.append(text)

    return "\n\n".join(formatted_chunks)

rag_prompt = ChatPromptTemplate.from_template(
    """
    You are an AI assistant answering questions using retrieved documents.

    Answer the question using ONLY the context provided below.

    Rules:
    1. Do not invent information.
    2. Cite the actual document name and page number from the context.
   Example: [EU AI Act (Reg. 2024/1689), p.115].
   Never use the literal word "source" as a citation.

    3. If the context does not contain the answer, say:
       "I don't know based on the sources."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

def answer(query, retrieve_fn=baseline_search):   # <- swap retrieve_fn to switch methods
    # TODO: retrieve -> format -> prompt | llm | StrOutputParser() -> return answer AND the results used
        results = retrieve_fn(query)

        context = format_context(results)

        chain = rag_prompt | llm | StrOutputParser()

        response = chain.invoke({
        "context": context,
        "question": query
    })

        return response, results

In [49]:
query = "What fines apply for using a prohibited AI practice?"

response, sources = answer(query)

print("QUESTION:", query)
print("\nANSWER:")
print(response)

QUESTION: What fines apply for using a prohibited AI practice?

ANSWER:
Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act (Reg. 2024/1689), p.115.0].


**Done when:** one query gives an answer with citations, and you can point to the exact chunk behind each claim.

In [50]:
import textwrap

print(textwrap.fill(response, width=100))

Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to
administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its
total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act
(Reg. 2024/1689), p.115.0].


In [47]:
import textwrap

print(textwrap.fill(response, width=100))

Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to
administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its
total worldwide annual turnover for the preceding financial year, whichever is higher [source,
p.115.0].


In [51]:
response, sources = answer(
    "What fines apply for using a prohibited AI practice?",
    retrieve_fn=crossencoder_rerank
)

print(textwrap.fill(response, width=100))

Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to
administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its
total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act
(Reg. 2024/1689), p.115.0].


In [52]:
response, sources = answer(
    "What is the population of Mars in 2026?",
    retrieve_fn=crossencoder_rerank
)

print(response)

I don't know based on the sources.


---
## Step 7: Evaluate: With vs. Without Reranking [YOUR TURN]

**Goal:** decide, with evidence, whether reranking was worth it.

**What to do:** for each test query, read the answers and the retrieved chunks yourself, then fill in the table. You are the judge: no automatic metric.

| Query | Method | Top-1 chunk relevant? (Y/N) | Answer correct? (Y/Partial/N) | Grounded in cited chunk? (Y/N) | Notes |
|---|---|---|---|---|---|
| Q1 | Baseline |N |Partial |N |Top chunk discusses AI literacy; p.51 citation does not support the claim. |
| Q1 | Reranked |N |Partial |N |More relevant passages retrieved, but p.51 citation does not support the claim. |
| Q2 | Baseline |Y |N |N/A |Retrieved related podcast content but could not answer the question. |
| Q2 | Reranked |Y |Y |Y|Correctly identified all seven principles, supported by EU AI Act p.8. |
| Q3 |Baseline  |Y |Y |Y |Correct explanation supported by podcast; citation metadata incomplete|
| Q3 |Reranked  |Y |Y |Y|Correct explanation supported by podcast; citation metadata incomplete.|
| Q4 |Baseline  |N |Y |Y|Correct answer supported by p.115, retrieved in third position. Top chunk p.51 does not contain the fines.|
| Q4 |Reranked  |Y |Y |Y|Reranking moved the relevant p.115 chunk to first position. Both methods produced the same correct answer.
| Q5 |Baseline  |Y |Partial|Y|Correctly identifies the prohibition and cites p.51, but incorrectly describes the practice as high-risk.|
| Q5 |Reranked  |Y |Y |Y|More precise explanation of prohibited social scoring, supported by Article 5 on p.51.|

**Then answer:**
- Did reranking change the answer, or only the order?
- What did it cost (extra calls, seconds per query)? Would a client pay for that?


## Step 7 – Final Evaluation

### 1. Did reranking change the answer or only the retrieval order?

Reranking improved retrieval quality and answer accuracy in some cases, but not consistently across all five test questions.

- **Q1 – Prohibited AI practices:** The reranked answer provided more information (three points instead of two), but both answers remained incomplete and had citation problems.
- **Q2 – Seven requirements for trustworthy AI:** Reranking significantly improved the result. The baseline could not answer the question, while the reranked method correctly identified all seven requirements with supporting evidence.
- **Q3 – Human-in-the-loop vs. human-on-the-loop:** Both methods produced correct, similar answers using relevant podcast content. Reranking provided no significant improvement.
- **Q4 – Fines for prohibited AI practices:** Both methods produced the same correct answer. However, reranking moved the relevant EU AI Act passage on page 115 from third position to first position.
- **Q5 – Social scoring:** Reranking improved answer accuracy. The baseline incorrectly described prohibited social scoring as high-risk, while the reranked answer explained the prohibition more accurately.

### 2. What did reranking cost, and would a client pay for it?

The initial timing comparison for Q1 produced the following results:

| Method | Response Time |
|---|---|
| Baseline | 4.09 seconds |
| Reranked | 4.39 seconds |
| **Additional time** | **0.30 seconds** |

Reranking requires an additional model-scoring step, which may increase computational costs and response time.

In this initial comparison, reranking added approximately 0.30 seconds. However, repeated measurements are needed to determine the average performance difference.

A client may consider this additional cost worthwhile when answer accuracy, retrieval relevance, and reliable source citations are important.

### 3. Overall Conclusion

The evaluation of five test questions showed that reranking can improve the performance of a Retrieval-Augmented Generation (RAG) system.

Reranking significantly improved answer quality in Q2 and Q5, improved retrieval order in Q4, and provided limited or no improvement in Q1 and Q3.

However, reranking did not eliminate all problems. Some answers remained incomplete, and citation accuracy was not always reliable.

**Recommendation:** Reranking appears worthwhile for applications where accurate, evidence-based answers are important. Before recommending it for production use, more test questions, repeated timing measurements, and improvements to citation handling are needed.


In [53]:
# YOUR TURN (optional helper): print baseline vs reranked answers side by side, and time each one
import time

q = TEST_QUERIES[0]

start = time.perf_counter()

baseline_answer, baseline_sources = answer(
    q,
    retrieve_fn=baseline_search
)

baseline_time = time.perf_counter() - start

print("QUESTION:", q)
print("\nBASELINE ANSWER:")
print(baseline_answer)
print("\nTIME:", round(baseline_time, 2), "seconds")

QUESTION: What AI practices are prohibited under the EU AI Act?

BASELINE ANSWER:
The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective or effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), p.51.0]. 

2. Practices that lead to economic or financial harms to consumers through unfair commercial practices, irrespective of whether they are implemented through AI systems or otherwise [EU AI Act (Reg. 2024/1689), p.9.0].

TIME: 4.09 seconds


In [54]:
start = time.perf_counter()

reranked_answer, reranked_sources = answer(
    q,
    retrieve_fn=crossencoder_rerank
)

reranked_time = time.perf_counter() - start

print("QUESTION:", q)
print("\nRERANKED ANSWER:")
print(reranked_answer)
print("\nTIME:", round(reranked_time, 2), "seconds")

QUESTION: What AI practices are prohibited under the EU AI Act?

RERANKED ANSWER:
The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), p.51.0].

2. AI-enabled manipulative techniques that persuade persons to engage in unwanted behaviours or deceive them by nudging them into decisions in a way that subverts and impairs their autonomy, decision-making, and free choices, particularly when significant harms to physical, psychological health, or financial interests are likely to occur [EU AI Act (Reg. 2024/1689), p.9.0]. 

3. Practices that contradict Union values of respect for 

In [55]:
print("BASELINE TOP CHUNK:")
print(baseline_sources[0][0].metadata)
print(baseline_sources[0][0].page_content[:500])

print("\nRERANKED TOP CHUNK:")
print(reranked_sources[0][0].metadata)
print(reranked_sources[0][0].page_content[:500])

BASELINE TOP CHUNK:
{'chunk_id': 'regulation-342', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
Article 4
AI literacy
Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
their staff and other persons dealing with the operation and use of AI systems on their behalf, taking into account their 
technical knowledge, experience, education and training and the context the AI systems are to be used in, and considering 
the persons or groups of persons on whom the AI systems are to be used.
CHAPTER II
PROHIBITED AI PRACTI

RERANKED TOP CHUNK:
{'chunk_id': 'regulation-84', 'page': 12.0, 'section': 'recital', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer 
protection law, and competition law, should not be affected by 

In [56]:
for name, sources in [
    ("BASELINE", baseline_sources),
    ("RERANKED", reranked_sources)
]:
    print("\n", name)

    for i, (doc, score) in enumerate(sources, 1):
        print(
            i,
            "Page:", doc.metadata.get("page"),
            "Text:", doc.page_content[:150]
        )


 BASELINE
1 Page: 51.0 Text: Article 4
AI literacy
Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
2 Page: 12.0 Text: (45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer 
protection law, and competition law,
3 Page: 9.0 Text: intention to distort behaviour where the distortion results from factors external to the AI system which are outside 
the control of the provider or t
4 Page: 45.0 Text: (b) prohibitions of certain AI practices;
(c) specific requirements for high-risk AI systems and obligations for operators of such systems;
(d) harmon
5 Page: 3.0 Text: operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in 
compliance with Union law, having t

 RERANKED
1 Page: 12.0 Text: (45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consume

In [57]:
import textwrap

print("BASELINE ANSWER:")
print(textwrap.fill(baseline_answer, width=100))

print("\nRERANKED ANSWER:")
print(textwrap.fill(reranked_answer, width=100))

BASELINE ANSWER:
The following AI practices are prohibited under the EU AI Act:  1. The placing on the market, the
putting into service, or the use of an AI system that deploys subliminal techniques beyond a
person’s consciousness or purposefully manipulative or deceptive techniques, with the objective or
effect of materially distorting the behavior of a person or a group of persons by appreciably
impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), p.51.0].   2.
Practices that lead to economic or financial harms to consumers through unfair commercial practices,
irrespective of whether they are implemented through AI systems or otherwise [EU AI Act (Reg.
2024/1689), p.9.0].

RERANKED ANSWER:
The following AI practices are prohibited under the EU AI Act:  1. The placing on the market, the
putting into service, or the use of an AI system that deploys subliminal techniques beyond a
person’s consciousness or purposefully manipulative or deceptive techniques, wi

In [58]:
print(TEST_QUERIES[1])

What are the seven requirements for trustworthy AI?


In [59]:

q = TEST_QUERIES[1]

baseline_answer, baseline_sources = answer(
    q,
    retrieve_fn=baseline_search
)

print("BASELINE ANSWER:")
print(textwrap.fill(baseline_answer, width=100))


BASELINE ANSWER:
I don't know based on the sources.


In [60]:

reranked_answer, reranked_sources = answer(
    q,
    retrieve_fn=crossencoder_rerank
)

print("RERANKED ANSWER:")
print(textwrap.fill(reranked_answer, width=100))


RERANKED ANSWER:
The seven requirements for trustworthy AI, as developed by the independent AI HLEG, include:  1.
Human agency and oversight 2. Technical robustness and safety 3. Privacy and data governance 4.
Transparency 5. Diversity, non-discrimination and fairness 6. Societal and environmental well-being
7. Accountability  These principles are intended to help ensure that AI is trustworthy and ethically
sound [EU AI Act (Reg. 2024/1689), p.8.0].


In [61]:

print("BASELINE TOP CHUNK:")
print(baseline_sources[0][0].metadata)
print(baseline_sources[0][0].page_content[:500])

print("\nRERANKED TOP CHUNK:")
print(reranked_sources[0][0].metadata)
print(reranked_sources[0][0].page_content[:500])


BASELINE TOP CHUNK:
{'source': 'data\\podcast.txt'}
make AI powerful to how do we make AI worthy of trust? And our mission for this deep dive is to figure out how to operationalize that. We aren't just talking philosophy here. No, no, no, we're looking at how you take an abstract concept like fairness and turn it into, I don't know, Python code. We've got three pillars, four principles and seven concrete requirements to get through. And a checklist that every developer should probably have tattooed on their arm. OK, let's start with a big picture

RERANKED TOP CHUNK:
{'chunk_id': 'regulation-50', 'page': 8.0, 'section': 'recital', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
(27) While the risk-based approach is the basis for a proportionate and effective set of binding rules, it is important to 
recall the 2019 Ethics guidelines for trustworthy AI developed by the independent AI HLEG appointed by the 
Commission. In those guidelines, the AI HLEG developed 

In [62]:
print(reranked_sources[0][0].page_content)

(27) While the risk-based approach is the basis for a proportionate and effective set of binding rules, it is important to 
recall the 2019 Ethics guidelines for trustworthy AI developed by the independent AI HLEG appointed by the 
Commission. In those guidelines, the AI HLEG developed seven non-binding ethical principles for AI which are 
intended to help ensure that AI is trustworthy and ethically sound. The seven principles include human agency and 
oversight; technical robustness and safety; privacy and data governance; transparency; diversity, non-discrimination 
and fairness; societal and environmental well-being and accountability. Without prejudice to the legally binding 
requirements of this Regulation and any other applicable Union law, those guidelines contribute to the design of 
coherent, trustworthy and human-centric AI, in line with the Charter and with the values on which the Union is


In [63]:
print(len(TEST_QUERIES))

5


In [64]:
print(TEST_QUERIES[2])

What is the difference between human-in-the-loop and human-on-the-loop?


In [65]:
q = TEST_QUERIES[2]

baseline_answer, baseline_sources = answer(
    q,
    retrieve_fn=baseline_search
)

print("BASELINE ANSWER:")
print(textwrap.fill(baseline_answer, width=100))

BASELINE ANSWER:
Human-in-the-loop (HITL) is the tightest control mechanism where the AI makes a suggestion, but a
human must approve every single action. In contrast, human-on-the-loop (HOTL) involves a human
monitoring the system and stepping in only if something appears to be unusual. This indicates a
gradient of control, with HITL requiring more direct human involvement compared to HOTL, where the
human's role is more supervisory [Unknown source, p.N/A].


In [66]:
reranked_answer, reranked_sources = answer(
    q,
    retrieve_fn=crossencoder_rerank
)

print("RERANKED ANSWER:")
print(textwrap.fill(reranked_answer, width=100))

RERANKED ANSWER:
Human-in-the-loop (HITL) is the tightest control mechanism where the AI makes a suggestion, but a
human must approve every single action. In contrast, human-on-the-loop (HOTL) involves a human
monitoring the system and stepping in only if something appears to be unusual. This distinction
highlights the varying levels of human oversight and control in AI systems [Unknown source, p.N/A].


In [67]:
print("BASELINE TOP CHUNK:")
print(baseline_sources[0][0].metadata)
print(baseline_sources[0][0].page_content[:500])

print("\nRERANKED TOP CHUNK:")
print(reranked_sources[0][0].metadata)
print(reranked_sources[0][0].page_content[:500])

BASELINE TOP CHUNK:
{'source': 'data\\podcast.txt'}
human in command. It's a gradient of control. Let's break that down. Human in the loop. That's the tightest control. The AI makes a suggestion, but a human has to click approve for every single action. But let me play doubles advocate here. If you have an AI processing 10,000 loan applications an hour, is human in the loop even real? That's a huge risk. Or is the human just a rubber stamp because they can't possibly review that fast? It's called automation bias. The human just trust the machine 

RERANKED TOP CHUNK:
{'source': 'data\\podcast.txt'}
human in command. It's a gradient of control. Let's break that down. Human in the loop. That's the tightest control. The AI makes a suggestion, but a human has to click approve for every single action. But let me play doubles advocate here. If you have an AI processing 10,000 loan applications an hour, is human in the loop even real? That's a huge risk. Or is the human just a rubber stamp be

In [68]:
print(baseline_sources[0][0].page_content)

human in command. It's a gradient of control. Let's break that down. Human in the loop. That's the tightest control. The AI makes a suggestion, but a human has to click approve for every single action. But let me play doubles advocate here. If you have an AI processing 10,000 loan applications an hour, is human in the loop even real? That's a huge risk. Or is the human just a rubber stamp because they can't possibly review that fast? It's called automation bias. The human just trust the machine too much or gets fatigued and just clicks. Yes, yes, yes. The guidelines warn about this. Just having a human sitting there isn't enough. They need to have the actual capacity to override the system. OK, so then you have human on the loop where the human monitors and just steps in if things look weird and human in command where the human decides when the system runs and when it dies, which brings up the stop button. Oh, yes, the big red button. The requirement explicitly mentions the capability


In [69]:
print(textwrap.fill(
    baseline_sources[0][0].page_content,
    width=100
))

human in command. It's a gradient of control. Let's break that down. Human in the loop. That's the
tightest control. The AI makes a suggestion, but a human has to click approve for every single
action. But let me play doubles advocate here. If you have an AI processing 10,000 loan applications
an hour, is human in the loop even real? That's a huge risk. Or is the human just a rubber stamp
because they can't possibly review that fast? It's called automation bias. The human just trust the
machine too much or gets fatigued and just clicks. Yes, yes, yes. The guidelines warn about this.
Just having a human sitting there isn't enough. They need to have the actual capacity to override
the system. OK, so then you have human on the loop where the human monitors and just steps in if
things look weird and human in command where the human decides when the system runs and when it
dies, which brings up the stop button. Oh, yes, the big red button. The requirement explicitly
mentions the capability


In [70]:
print(TEST_QUERIES[3])

What fines apply for using a prohibited AI practice?


In [84]:
q = TEST_QUERIES[3]

baseline_answer, baseline_sources = answer(
    q,
    retrieve_fn=baseline_search
)

print("BASELINE ANSWER:")
print(textwrap.fill(baseline_answer, width=100))

BASELINE ANSWER:
Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to
administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its
total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act
(Reg. 2024/1689), p.115.0].


In [85]:
reranked_answer, reranked_sources = answer(
    q,
    retrieve_fn=crossencoder_rerank
)

print("RERANKED ANSWER:")
print(textwrap.fill(reranked_answer, width=100))

RERANKED ANSWER:
Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to
administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its
total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act
(Reg. 2024/1689), p.115.0].


In [73]:
print("BASELINE TOP CHUNK:")
print(baseline_sources[0][0].metadata)
print(textwrap.fill(baseline_sources[0][0].page_content, width=100))

print("\nRERANKED TOP CHUNK:")
print(reranked_sources[0][0].metadata)
print(textwrap.fill(reranked_sources[0][0].page_content, width=100))

BASELINE TOP CHUNK:
{'chunk_id': 'regulation-342', 'page': 51.0, 'section': 'article', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their
best extent, a sufficient level of AI literacy of  their staff and other persons dealing with the
operation and use of AI systems on their behalf, taking into account their  technical knowledge,
experience, education and training and the context the AI systems are to be used in, and considering
the persons or groups of persons on whom the AI systems are to be used. CHAPTER II PROHIBITED AI
PRACTICES Article 5 Prohibited AI practices 1. The following AI practices shall be prohibited: (a)
the placing on the market, the putting into service or the use of an AI system that deploys
subliminal techniques beyond  a person’s consciousness or purposefully manipulative or deceptive
techniques, with the objective, or the effect of  material

In [86]:

for rank, (doc, score) in enumerate(reranked_sources, start=1):
    print(
        "Rank:", rank,
        "| Page:", doc.metadata.get("page"),
        "| Chunk ID:", doc.metadata.get("chunk_id"),
        "| Score:", score
    )


Rank: 1 | Page: 115.0 | Chunk ID: regulation-665 | Score: 4.258953
Rank: 2 | Page: 117.0 | Chunk ID: regulation-673 | Score: 4.09798
Rank: 3 | Page: 108.0 | Chunk ID: regulation-635 | Score: 2.5041847
Rank: 4 | Page: 51.0 | Chunk ID: regulation-342 | Score: 2.306462
Rank: 5 | Page: 9.0 | Chunk ID: regulation-58 | Score: 1.0243292


In [74]:
for doc, score in baseline_sources:
    print(doc.metadata.get("page"), doc.metadata.get("chunk_id"))

51.0 regulation-342
118.0 regulation-677
115.0 regulation-665
117.0 regulation-673
117.0 regulation-675


In [75]:
print(TEST_QUERIES[4])

Is social scoring of citizens allowed?


In [76]:
q = TEST_QUERIES[4]

baseline_answer, baseline_sources = answer(
    q,
    retrieve_fn=baseline_search
)

print("BASELINE ANSWER:")
print(textwrap.fill(baseline_answer, width=100))

BASELINE ANSWER:
Social scoring of citizens is prohibited if it leads to detrimental or unfavourable treatment of
natural persons or groups based on their social behaviour or inferred personal characteristics. Such
practices may violate rights to dignity and non-discrimination, and are classified as high-risk
under the regulation [EU AI Act (Reg. 2024/1689), p.51.0].


In [77]:
reranked_answer, reranked_sources = answer(
    q,
    retrieve_fn=crossencoder_rerank
)

print("RERANKED ANSWER:")
print(textwrap.fill(reranked_answer, width=100))

RERANKED ANSWER:
Social scoring of citizens is prohibited if it leads to detrimental or unfavourable treatment of
natural persons or groups in social contexts unrelated to the original data context or if such
treatment is unjustified or disproportionate to their social behaviour. This is outlined in the EU
AI Act [EU AI Act (Reg. 2024/1689), p.51.0].


In [78]:
print("BASELINE TOP CHUNK:")
print(baseline_sources[0][0].metadata)
print(textwrap.fill(baseline_sources[0][0].page_content, width=100))

print("\nRERANKED TOP CHUNK:")
print(reranked_sources[0][0].metadata)
print(textwrap.fill(reranked_sources[0][0].page_content, width=100))

BASELINE TOP CHUNK:
{'chunk_id': 'regulation-61', 'page': 9.0, 'section': 'recital', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'source_type': 'regulation'}
multiple data points related to their social behaviour in multiple contexts or known, inferred or
predicted personal  or personality characteristics over certain periods of time. The social score
obtained from such AI systems may lead  to the detrimental or unfavourable treatment of natural
persons or whole groups thereof in social contexts, which  are unrelated to the context in which the
data was originally generated or collected or to a detrimental treatment that  is disproportionate
or unjustified to the gravity of their social behaviour. AI systems entailing such unacceptable
scoring practices and leading to such detrimental or unfavourable outcomes should therefore be
prohibited. That  prohibition should not affect lawful evaluation practices of natural persons that
are carried out for a specific purpose  in accordance with

In [79]:
print("BASELINE SOURCES:")
for doc, score in baseline_sources:
    print(doc.metadata.get("page"), doc.metadata.get("chunk_id"))

print("\nRERANKED SOURCES:")
for doc, score in reranked_sources:
    print(doc.metadata.get("page"), doc.metadata.get("chunk_id"))

BASELINE SOURCES:
9.0 regulation-61
9.0 regulation-60
51.0 regulation-344
None None
16.0 regulation-113

RERANKED SOURCES:
9.0 regulation-60
9.0 regulation-61
None None
51.0 regulation-344
16.0 regulation-113


In [80]:
for doc, score in reranked_sources:
    if doc.metadata.get("page") == 51:
        print(textwrap.fill(doc.page_content, width=100))

(c) the placing on the market, the putting into service or the use of AI systems for the evaluation
or classification of natural  persons or groups of persons over a certain period of time based on
their social behaviour or known, inferred or  predicted personal or personality characteristics,
with the social score leading to either or both of the following: (i) detrimental or unfavourable
treatment of certain natural persons or groups of persons in social contexts that are  unrelated to
the contexts in which the data was originally generated or collected; (ii) detrimental or
unfavourable treatment of certain natural persons or groups of persons that is unjustified or
disproportionate to their social behaviour or its gravity; (d) the placing on the market, the
putting into service for this specific purpose, or the use of an AI system for making risk
assessments of natural persons in order to assess or predict the risk of a natural person committing
a criminal offence,


---
## Step 8: Write `lab_proof.md` [YOUR TURN]

Create `lab_proof.md` in your repo (this is the **entry point for your reviewer**). Use this template:

```markdown
# Lab Proof: First match ≠ right match

## 1. What I built
One or two sentences, plus a link to this notebook.

## 2. Query
"..."

## 3. Retrieved evidence
| Rank | Source | Page / chunk_id | Score | Snippet |
|---|---|---|---|---|

## 4. Final answer
(paste the answer, including its citations)

## 5. Trace
Which claim in the answer comes from which chunk?

## 6. Baseline vs reranked
What changed? Was it worth the cost?

## 7. Failure or limitation
Where could this system give a plausible but unsupported answer?
```

**Before you submit:**
- [ ] Only files for this lab are in the repo
- [ ] No API keys, `.env` or tokens are committed
- [ ] `lab_proof.md` is the entry point
- [ ] A reviewer can trace **one** answer back to the exact retrieved chunk

---
### Extensions (optional)
- **Hybrid search:** combine BM25 keyword search with vector search, then rerank
- **Cost analysis:** cost per query for baseline vs LLM scoring vs cross-encoder
- **Failure test:** ask something neither source covers. Does your pipeline say "I don't know"?